# 2Dify AI — READY Private Colab

Upload a video → Lucy V2V converts the scene to hand-drawn 2D → original audio is removed → silent MP4 is saved to Google Drive.

This notebook validates the worker before starting. A remote GPU is required; free GPU availability is not guaranteed.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/2Dify')
ROOT.mkdir(parents=True, exist_ok=True)
print('Drive workspace:', ROOT)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U gradio_client requests
print('Dependencies installed.')

In [ ]:
REPO = Path('/content/2dify-repo')
if not (REPO / '.git').exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git','clone','-q','https://github.com/mukz3467/Multi-tool-Ai-.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','-q','origin/main'], check=True)
WORKER = REPO / '2dify' / '2dify_worker.py'
if not WORKER.is_file():
    raise FileNotFoundError(f'Worker not found: {WORKER}')
subprocess.run([sys.executable,'-m','py_compile',str(WORKER)], check=True)
print('Worker syntax check: OK')
print('Worker:', WORKER)

## 1) Select a video

For a new job choose **U** and upload. After a Colab restart, select the same Drive video to resume from saved chunks.

In [ ]:
VIDEO_EXTS = {'.mp4','.mov','.mkv','.webm','.avi','.m4v'}
videos = sorted([p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS], key=lambda p: p.name.lower())
if videos:
    print('Videos already in Drive:')
    for i, p in enumerate(videos):
        print(f'  {i}: {p.name}')
else:
    print('No videos found in Drive yet.')

choice = input('Choose video number, or type U to upload: ').strip()
if choice.lower() == 'u':
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No video uploaded.')
    name = next(iter(uploaded))
    src = Path(name)
    if src.suffix.lower() not in VIDEO_EXTS:
        raise ValueError(f'Unsupported video format: {src.suffix}')
    INPUT = ROOT / src.name
    shutil.copy2(src, INPUT)
elif choice.isdigit() and 0 <= int(choice) < len(videos):
    INPUT = videos[int(choice)]
else:
    raise ValueError('Invalid selection. Run this cell again and choose a listed number or U.')

OUTPUT = ROOT / f'{INPUT.stem}_2dify_silent.mp4'
print('Input :', INPUT)
print('Output:', OUTPUT)
print('Checkpoint folder:', OUTPUT.parent / (OUTPUT.stem + '.2dify_chunks'))

## 2) Convert

Each Lucy window is about 3.4 seconds. Completed windows are stored in Google Drive and reused after an interruption. **Run this cell only after selecting a video.**

In [ ]:
if not INPUT.exists():
    raise FileNotFoundError(INPUT)
cmd = [sys.executable, str(WORKER), str(INPUT), str(OUTPUT)]
print('Starting conversion...')
subprocess.run(cmd, check=True)
if not OUTPUT.exists() or OUTPUT.stat().st_size <= 1024:
    raise RuntimeError('Conversion ended without a valid output file.')
print('DONE:', OUTPUT)
print('Size (MB):', round(OUTPUT.stat().st_size / 1024 / 1024, 2))

In [ ]:
if OUTPUT.exists() and OUTPUT.stat().st_size > 1024:
    files.download(str(OUTPUT))
else:
    print('No final file to download. Keep the Drive checkpoint folder and resume the same job.')